# Transformer Architecture: Putting it All Together

Reach for this when you need: 
- Reference for full Transformer Encoder/Decoder layout.
- To understand Feed-Forward Networks (FFN) and LayerNorm.
- Implementation of a skeletal Transformer Block.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. The Transformer Block

A standard Transformer block consists of: 
1. Multi-Head Attention (MHA)
2. Add & Norm (Residual + LayerNorm)
3. Feed-Forward Network (FFN)
4. Add & Norm

| Layer | Description | Purpose |
| :--- | :--- | :--- |
| `LayerNorm` | Normalizes features across the embedding dimension | Prevents activation explosion |
| `Residual` | $x + Sublayer(x)$ | Stabilizes gradient flow through many blocks |
| `FFN` | Position-wise linear transformations | Adds non-linear complexity after attention |

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.mha = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.ReLU(),
            nn.Linear(4 * embed_dim, embed_dim)
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Multi-Head Attention + Residual
        attn_out, _ = self.mha(x, x, x)
        x = self.norm1(x + self.dropout(attn_out))
        
        # Feed-Forward + Residual
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x

## 2. Encoder vs Decoder

| Architecture | Use Case | Key Feature |
| :--- | :--- | :--- |
| **Encoder Only** | Classification, NER (e.g. BERT) | Bidirectional context |
| **Decoder Only** | Generation (e.g. GPT-4, Llama) | Causal masking (no looking ahead) |
| **Encoder-Decoder** | Translation (e.g. T5, BART) | Cross-attention between Enc and Dec |

In [ ]:
# PyTorch helper for full architectures
encoder_layer = nn.TransformerEncoderLayer(d_model=512, nhead=8, batch_first=True)
transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=6)

### Common Pitfalls
- **Pre-norm vs Post-norm**: Modern LLMs (Llama) use 'Pre-norm' (LayerNorm before the MHA/FFN) for better stability in very deep models. PyTorch defaults to 'Post-norm'.
- **LayerNorm vs BatchNorm**: DO NOT use BatchNorm in Transformers. Sequences are highly variable; LayerNorm (normalizing across features) works best for text.
- **Sequence Length**: Attention is memory-intensive ($N^2$); ensure sequences aren't so long they trigger OOM on your GPU cached blocks.

### Key Takeaways
- The Transformer is a stack of modular blocks that transform representations through attention and non-linearities.
- LayerNorm is essential for reliable convergence in deep architectures.
- Residual connections act as gradient highways, enabling training of models with hundreds of layers.